# Prefix Tuning 交互教学

配套 lecture：[`../lectures/01-prefix-tuning.md`](../lectures/01-prefix-tuning.md)  
配套论文：[`../papers/01-prefix-tuning-2021.pdf`](../papers/01-prefix-tuning-2021.pdf)

本 notebook 演示：
1. 环境检查
2. 手写 minimal 版构造（含 reparam MLP）
3. peft 调包版
4. 数值一致性验证（弱一致）
5. 消融对比：去掉 reparam 看参数量变化
6. 思考题

## 1. 环境检查

In [ ]:
import sys
import torch, transformers, peft
print(f'Python: {sys.version.split()[0]}')
print(f'torch:        {torch.__version__}')
print(f'transformers: {transformers.__version__}')
print(f'peft:         {peft.__version__}')

## 2. 手写 minimal 版

公式 (2): $\mathrm{head}_h^{(\ell)} = \mathrm{Attn}(Q, [P^K; K], [P^V; V])$

实现方式：用 `transformers.DynamicCache` 把可训练 prefix 注入到每层的 past_key_values。

In [ ]:
from pathlib import Path
src_dir = (Path.cwd().parent / 'src').resolve()
sys.path.insert(0, str(src_dir))

from prefix_tuning_minimal import PrefixTuningGPT2
from common import print_param_summary

torch.manual_seed(42)
model = PrefixTuningGPT2(prefix_length=10, mid_dim=512)
print_param_summary(model, 'minimal (p=10, mid=512)')
# 预期：可训练 ≈ 9.8M（主要由 MLP 贡献）

In [ ]:
# 拆解：P_low (低维 prefix) 与 MLP 的参数量分别多少？
n_low = model.P_low.numel()
n_mlp = sum(p.numel() for p in model.reparam.parameters())
print(f'P_low (P\'): {n_low:,} 参数 ← 公式 (3) 的低维 prefix')
print(f'MLP_φ:      {n_mlp:,} 参数 ← 公式 (3) 的 reparameterization MLP')
print(f'合计:       {n_low + n_mlp:,}')

## 3. peft 调包版

In [ ]:
from prefix_tuning_peft import build_peft_model

torch.manual_seed(42)
peft_model = build_peft_model(prefix_length=10, mid_dim=512)
print_param_summary(peft_model, 'peft (p=10, mid=512)')

print('\npeft 内部参数清单：')
for name, p in peft_model.named_parameters():
    if p.requires_grad:
        print(f'  {name}: {tuple(p.shape)} = {p.numel():,}')

## 4. 数值一致性验证（弱一致）

由于 reparam 涉及随机初始化，不强求 logits bit 精确；仅验证形状一致 + 参数量同量级。

In [ ]:
tests_dir = (Path.cwd().parent / 'src' / 'tests').resolve()
sys.path.insert(0, str(tests_dir))
from test_prefix_consistency import test_shape_and_param_order
test_shape_and_param_order()

## 5. 消融：去掉 reparam

论文 Table 5 显示：去掉 reparam（直接训练每层 KV 大张量），BLEU 掉 3.2。
我们这里只对比参数量与前向能跑通。

In [ ]:
torch.manual_seed(42)
m_no_reparam = PrefixTuningGPT2(prefix_length=10, use_reparam=False)
print_param_summary(m_no_reparam, 'no reparam (p=10)')
# 预期：184,320 = L * p * 2 * d = 12 * 10 * 2 * 768
print(f'\n核算：12 * 10 * 2 * 768 = {12*10*2*768:,}')

# 前向
tok = m_no_reparam.tokenizer
enc = tok('test', return_tensors='pt', padding=True)
with torch.no_grad():
    out = m_no_reparam(enc['input_ids'], enc['attention_mask'])
print(f'logits shape: {tuple(out.logits.shape)}')

## 6. 思考题

1. **参数量计算**：当前 `p=10, mid=512, L=12, d=768`，训练参数 9.8M。如果固化 MLP 输出（推理时丢 MLP），参数变多少？
2. **DynamicCache vs tuple**：transformers 4.x 用 tuple of (K, V) 元组，5.x 用 DynamicCache。如果代码迁移到 4.x，需要改哪里？
3. **mid_dim 影响**：把 `mid_dim` 从 512 改成 100，参数量降多少？训练效果会受影响吗？预测并验证。
4. **Embedding-only 退化**：如果让 prefix 只作用于第 1 层（不是每层），参数省了多少？这是不是就是 Prompt Tuning？为什么不完全是？
5. **与 Prompt Tuning 比较**：本 notebook 是 9.8M 参数，[`02-prompt-tuning.ipynb`](02-prompt-tuning.ipynb) 是 7.7K 参数。在 GPT-2 base（117M）这个小模型上，哪种更有可能在 toy 任务上 loss 下降快？为什么？

In [ ]:
# 提示：思考题 1
L, p, d = 12, 10, 768
推理参数 = L * p * 2 * d
print(f'推理时参数（每层 K、V 直接存）= {推理参数:,}')
print(f'训练时参数（含 MLP）= 9,857,024')
print(f'压缩比 = {9_857_024 / 推理参数:.1f}x')